# Python 101 - Solutions
## Chapter IV

---

**For teaching assistants.** This notebook mirrors the exercises in
`../python101_04.ipynb` one for one. Most solutions end with an `assert`, so
running the whole notebook top to bottom is also a self-test: if it runs clean,
every solution still works.

There is usually more than one right answer - if a student's version passes the
same `assert`, it is correct.

In [ ]:
# Run from the chapter folder, so that `helpers`, `./data/...` and `./pics/...`
# resolve exactly the way they do in the lecture notebooks.
import os
import sys

if os.path.basename(os.getcwd()) == 'solutions':
    os.chdir('..')
sys.path.insert(0, os.getcwd())

print('working directory:', os.getcwd())

In [ ]:
import random

from helpers import *

BASE = './data/'

## Warming up: Deus ex Python

### 1. Basic stats

In [ ]:
def stat(numbers):
    return {
        'mean': sum(numbers) / len(numbers),
        'min': min(numbers),
        'max': max(numbers),
    }


lon = [random.random() for _ in range(30)]
print(stat(lon))

assert stat([1, 2, 3]) == {'mean': 2.0, 'min': 1, 'max': 3}

### 2. Running mean (window size 3)

`range(len(numbers) - window + 1)` is the bit worth stopping on: with a window of 3 over 5 numbers you get 3 results, not 5.

In [ ]:
def running_mean(numbers, window=3):
    means = []
    for start in range(len(numbers) - window + 1):
        chunk = numbers[start:start + window]
        means.append(sum(chunk) / window)
    return means


print(running_mean([1, 2, 3, 4, 5]))

assert running_mean([1, 2, 3, 4, 5]) == [2.0, 3.0, 4.0]
assert len(running_mean(lon)) == len(lon) - 2

### 3. Rename the mismatching subtitles

`download_series(mismatch=True)` gives subtitles with random separators, random release-group noise and random capitalisation. The season/episode marker is the only thing both names share, so match on that.

In [ ]:
import shutil

shutil.rmtree('super_series', ignore_errors=True)
print(download_series(seasons=2, episodes=3, mismatch=True))


def fix_subtitles(directory):
    files = list_files(directory)
    videos = [f for f in files if f.lower().endswith('.avi')]
    subtitles = [f for f in files if f.lower().endswith('.srt')]

    # marker (S01E02) -> the name the subtitle should have
    wanted = {}
    for video in videos:
        marker = find_episode_number(video)
        wanted[marker.upper()] = video[:-4] + '.srt'

    renamed = 0
    for subtitle in subtitles:
        marker = find_episode_number(subtitle)
        if marker is None:
            continue
        correct = wanted.get(marker.upper())
        if correct and correct != subtitle:
            rename_subtitle(subtitle, correct, directory)
            renamed += 1
    return renamed


print('renamed:', fix_subtitles('super_series'))

# every video now has a subtitle with exactly the same name
files = list_files('super_series')
videos = {f[:-4] for f in files if f.endswith('.avi')}
subs = {f[:-4] for f in files if f.endswith('.srt')}
assert videos == subs, (videos - subs, subs - videos)
print('all', len(videos), 'episodes matched')

shutil.rmtree('super_series', ignore_errors=True)

## 1. Write a matrix into a csv file

In [ ]:
matrix = [[random.random() for _ in range(10)] for _ in range(10)]

path = export_to_csv(BASE + 'matrix.csv', matrix)
print('written to', path)

back = import_from_csv(BASE + 'matrix.csv')
assert len(back) == 10 and len(back[0]) == 10
# csv stores text, so everything comes back as a string
assert isinstance(back[0][0], str)
assert abs(float(back[0][0]) - matrix[0][0]) < 1e-9

## 2. Write our own fake `download` function

A cut-down version of `download_series` - enough to make files to play with.

In [ ]:
import os


def download(name='my_series', seasons=2, episodes=3):
    os.makedirs(name, exist_ok=True)
    created = 0
    for extension in ('avi', 'srt'):
        for season in range(1, seasons + 1):
            for episode in range(1, episodes + 1):
                filename = f'{name}.S{season:02d}E{episode:02d}.{extension}'
                with open(os.path.join(name, filename), 'w', encoding='utf-8') as handle:
                    handle.write(filename)
                created += 1
    return f'{created} files created in {name}/'


print(download())
assert len(list_files('my_series')) == 12
shutil.rmtree('my_series', ignore_errors=True)

## 3. Merge the matching rows

Two things to point out:
- `matching.csv` is **semicolon** separated, so pass `delimiter=';'`
- the header in the file is `ID;NAME;VAL2;VAL3`, not `val1;val2` as the exercise text says. Reading the actual file rather than trusting the description is half the lesson.

In [ ]:
rows = import_from_csv(BASE + 'matching.csv', delimiter=';')
header, records = rows[0], rows[1:]
print(header)
print(records[:3])

merged = {}
order = []
for record_id, name, first, second in records:
    if record_id not in merged:
        merged[record_id] = [record_id, name, int(first), int(second)]
        order.append(record_id)
    else:
        merged[record_id][1] += ' & ' + name
        merged[record_id][2] += int(first)
        merged[record_id][3] += int(second)

result = [header] + [merged[record_id] for record_id in order]
export_to_csv(BASE + 'merged.csv', result, delimiter=';')

for row in result[:4]:
    print(row)

assert merged['1'] == ['1', 'Neo & Trinity', 15, 76]
assert merged['2'] == ['2', 'Bud & Terence', 14, 55]
assert len(merged) == len({r[0] for r in records})

## 4. Word counting

`sorted(..., key=..., reverse=True)` is the part they need `help(sorted)` for. `collections.Counter` does the same job in one line - show it afterwards, not before.

In [ ]:
def count_words(filename, n=5):
    with open(filename, encoding='utf-8') as handle:
        text = handle.read()

    counts = {}
    for word in text.lower().split():
        word = word.strip('.,!?:;()\'"')
        if not word:
            continue
        counts[word] = counts.get(word, 0) + 1

    return sorted(counts.items(), key=lambda pair: pair[1], reverse=True)[:n]


top = count_words(BASE + 'text.txt', n=5)
for word, count in top:
    print(f'{word:14s} {count}')

# data/text.txt is the Zen of Python: "is better than" repeats a lot
words = [word for word, _ in top]
assert 'better' in words and 'than' in words
assert top[0][1] >= top[-1][1]     # sorted descending

# ...and the one-liner, once they have met collections:
import collections
with open(BASE + 'text.txt', encoding='utf-8') as handle:
    quick = collections.Counter(handle.read().lower().split()).most_common(5)
print(quick)

## 5-6. Encrypt and decrypt a file

`encrypt(text, strength=n)` at level 1 inserts `n - 1` junk characters after every real one - so decrypting is just `text[::n]`. Same idea as the slicing exercise in chapter II, now applied to a whole file.

In [ ]:
def encrypt_file(filename, strength=4):
    with open(filename, encoding='utf-8') as handle:
        text = handle.read()

    stem, _, extension = filename.rpartition('.')
    target = f'{stem}_encrypted.{extension}'

    with open(target, 'w', encoding='utf-8') as handle:
        handle.write(encrypt(text, strength=strength))
    return target


def decrypt_file(filename, strength=4):
    with open(filename, encoding='utf-8') as handle:
        text = handle.read()

    stem = filename.replace('_encrypted', '')
    stem, _, extension = stem.rpartition('.')
    target = f'{stem}_decrypted.{extension}'

    with open(target, 'w', encoding='utf-8') as handle:
        handle.write(text[::strength])
    return target


encrypted = encrypt_file(BASE + 'text.txt', strength=4)
decrypted = decrypt_file(encrypted, strength=4)
print(encrypted, '->', decrypted)

original = open(BASE + 'text.txt', encoding='utf-8').read()
assert open(decrypted, encoding='utf-8').read() == original
print('round-trip exact')

for path in (encrypted, decrypted, BASE + 'matrix.csv', BASE + 'merged.csv'):
    os.remove(path)

---
## Further exercises: medians and modes

In [ ]:
numbers = [random.random() for _ in range(30)]
sorted_numbers = sorted(numbers)

# 1. the median of an already sorted list, inline
middle = len(sorted_numbers) // 2
if len(sorted_numbers) % 2:
    median_value = sorted_numbers[middle]
else:
    median_value = (sorted_numbers[middle - 1] + sorted_numbers[middle]) / 2
print(median_value)


# 2-3. as a function. Sorting inside the function makes it work for both
#      sorted and unsorted input, which is exercise 3 for free.
def median(numbers):
    values = sorted(numbers)
    middle = len(values) // 2
    if len(values) % 2:
        return values[middle]
    return (values[middle - 1] + values[middle]) / 2


assert median([3, 1, 2]) == 2               # odd length
assert median([4, 1, 3, 2]) == 2.5          # even length
assert median(numbers) == median(sorted_numbers)
print(median(numbers))

### 4. Moving median

In [ ]:
def moving_median(numbers, window=3):
    return [median(numbers[start:start + window])
            for start in range(len(numbers) - window + 1)]


print(moving_median([1, 5, 2, 8, 3]))

assert moving_median([1, 5, 2, 8, 3]) == [2, 5, 3]
assert len(moving_median(numbers)) == len(numbers) - 2

### 5. Mode

Note the trap: with 30 random floats **every** value appears exactly once, so the mode is meaningless. Good moment to ask what the mode is actually for.

In [ ]:
def mode(numbers):
    counts = {}
    for number in numbers:
        counts[number] = counts.get(number, 0) + 1

    highest = max(counts.values())
    return [number for number, count in counts.items() if count == highest]


assert mode([1, 2, 2, 3]) == [2]
assert sorted(mode([1, 1, 2, 2])) == [1, 2]      # two modes
assert len(mode(numbers)) == len(numbers)        # all floats unique - useless!
print('mode of 3 random floats:', len(mode(numbers)), 'values tie')

## Bonus: bubble sort

`numbers[:]` makes a copy, so the function does not scramble the caller's list - which the `assert` in the notebook relies on.

In [ ]:
def bubble_sort(numbers):
    values = numbers[:]
    for end in range(len(values) - 1, 0, -1):
        swapped = False
        for i in range(end):
            if values[i] > values[i + 1]:
                values[i], values[i + 1] = values[i + 1], values[i]
                swapped = True
        if not swapped:
            break
    return values


assert sorted(numbers) == bubble_sort(numbers), 'Error, sorting mismatch!'
assert bubble_sort([3, 1, 2]) == [1, 2, 3]
assert bubble_sort([]) == [] and bubble_sort([1]) == [1]
print('bubble sort agrees with sorted() on', len(numbers), 'numbers')

---
## Extra exercises (the old homework)

### 7. Is it a prime?

`isinstance(n, bool)` has to be excluded explicitly, because in python `True` **is** an `int`. That is a genuinely surprising fact worth showing.

In [ ]:
def is_prime(number):
    if isinstance(number, bool) or not isinstance(number, int):
        print(f'ERROR: {number!r} is not a whole number!')
        return False
    if number < 2:
        return False
    divisor = 2
    while divisor * divisor <= number:
        if number % divisor == 0:
            return False
        divisor += 1
    return True


assert [n for n in range(20) if is_prime(n)] == [2, 3, 5, 7, 11, 13, 17, 19]
assert not is_prime(1) and not is_prime(0) and not is_prime(-7)
assert not is_prime(2.5) and not is_prime('7')
assert not is_prime(True)      # True == 1, but it is not a number we accept

### 8. The first N primes

In [ ]:
def first_primes(n):
    primes = []
    candidate = 2
    while len(primes) < n:
        if is_prime(candidate):
            primes.append(candidate)
        candidate += 1
    return primes


print(first_primes(10))

assert first_primes(5) == [2, 3, 5, 7, 11]
assert len(first_primes(25)) == 25
assert first_primes(0) == []

### 9. Factorial

In [ ]:
def factorial(number):
    if isinstance(number, bool) or not isinstance(number, int) or number < 0:
        print(f'ERROR: {number!r} is not a non-negative whole number!')
        return None

    result = 1
    for value in range(2, number + 1):
        result *= value
    return result


print(factorial(10))

assert factorial(0) == 1 and factorial(1) == 1
assert factorial(5) == 120
assert factorial(10) == 3628800
assert factorial(2.5) is None and factorial(-1) is None

### 10. Extract one attribute from a list of dicts

In [ ]:
users = [
    {'name': 'Peter Parker',  'alterego': 'Spider-man',     'email': 'peter.parker@daily.com',   'occupation': 'photographer'},
    {'name': 'Harry Osborn',  'alterego': 'Green Goblin',   'email': 'harry@osborne.com',        'occupation': 'CEO'},
    {'name': 'Otto Octavius', 'alterego': 'Doctor Octopus', 'email': 'otto.octavius@atomki.com', 'occupation': 'nuclear physicist'},
]


def extract(users, attribute):
    return [user[attribute] for user in users if attribute in user]


print(extract(users, 'occupation'))
print(extract(users, 'age'))

assert extract(users, 'occupation') == ['photographer', 'CEO', 'nuclear physicist']
assert extract(users, 'age') == []       # missing keys are skipped, not an error

### 11. Merge parallel lists into a list of dicts

`zip(*lists)` is the neat way: it walks all the lists side by side. Show the explicit index version too if that lands better.

In [ ]:
names = ['Peter Parker', 'Harry Osborn', 'Otto Octavius']
alteregos = ['Spider-man', 'Green Goblin', 'Doctor Octopus']
ages = [21, 21, 45]


def merge(lists, names):
    if len(lists) != len(names):
        print('ERROR: parameter length mismatch!')
        return None

    return [dict(zip(names, values)) for values in zip(*lists)]


print(merge(lists=[names, ages], names=['name', 'age']))
print(merge(lists=[names], names=['name']))
print(merge(lists=[names], names=['name', 'age']))

assert merge(lists=[names, ages], names=['name', 'age'])[0] == \
    {'name': 'Peter Parker', 'age': 21}
assert merge(lists=[names], names=['name'])[2] == {'name': 'Otto Octavius'}
assert merge(lists=[names], names=['name', 'age']) is None
assert len(merge(lists=[names, alteregos, ages],
                 names=['name', 'alterego', 'age'])) == 3